## Training
Prepapre dataset with the prepare_dataset notebook, before running this one.

In [ ]:
from torch.utils.data import DataLoader
# from tqdm import tqdm
from time import time, sleep
import numpy as np
import os
import warnings; warnings.filterwarnings("ignore")
from utils import *
  
# DEV = torch.device(CPU) # for debugging, use cpu
os.makedirs(f"{SAVE_DIR}", exist_ok=True)
print(f'DEV: {DEV}, has_screen: {LOCAL}, job id: {JOBID}')

# copy the python training to the directory (for cluster) (for local, it fails silently)
os.system(f"cp train.py {SAVE_DIR}/train.py")
os.system(f"cp utils.py {SAVE_DIR}/utils.py")

In [ ]:
EPOCHS = 12 if LOCAL else 50 # number of epochs 100 <-
# BATCH_SIZE = 128 # 128 
BATCH_SIZE = 4 if LOCAL else 64 # 64 <-
print(f'EPOCHS: {EPOCHS}, BATCH_SIZE: {BATCH_SIZE}')

# LOAD_PRETRAINED = f'{BEST_MODEL_DIR}/best_{FX}.pth' if LOCAL else None # pretrained model
LOAD_PRETRAINED = None # Set it to None if you don't want to load pretrained model
# LOAD_PRETRAINED = 'data/3067354/best_Fx.pth'

In [ ]:
# schedulers
from numpy import concatenate as cat, linspace as linsp, logspace as logsp
### learning rate
# LR = 3e-4*logsp(0, -2, EPOCHS) 
# LR = 3e-3*np.ones(EPOCHS) 
# LR = 3e-3*logsp(0, -2, EPOCHS)  
# LR = 1e-3*np.ones(EPOCHS) if LOCAL else 1e-3*logsp(0, -2, EPOCHS) # <-
LR = 5e-3*logsp(0, -2, EPOCHS) # <- [good]
# LR = 5e-4*np.ones(EPOCHS)
# LR = 5e-3*np.ones(EPOCHS)

### loss ratios 
## Fx
R_FX = np.ones(EPOCHS) 
# R_FX = 1.0 * logsp(0, -4, EPOCHS) # <- for Iy estimate

## Iy
R_IY = 1e-5 * np.ones(EPOCHS) 
# R_IY  = 1e-5 * logsp(0, -4, EPOCHS) # <- plot/good
# R_IY = np.zeros(EPOCHS) # <- best for Fx

## Br Bz
# R_BRZ = 0.1 * np.ones(EPOCHS) 
R_BRZ = np.zeros(EPOCHS) 

## PINN
R_PIN = 0.1 * np.ones(EPOCHS) 
# R_PIN = np.zeros(EPOCHS) 

ACTIVE_LOSSES = [np.sum(l_profile) > 0 for l_profile in [R_FX, R_IY, R_BRZ, R_PIN]]

In [ ]:
#checks
if LOAD_PRETRAINED is not None: assert os.path.exists(LOAD_PRETRAINED), "Pretrained model does not exist"
assert os.path.exists(TRAIN_DS_PATH), "Training dataset does not exist"
assert os.path.exists(EVAL_DS_PATH), "Evaluation dataset does not exist"
assert os.path.exists(SAVE_DIR), "Save directory does not exist"
assert len(LR) == EPOCHS, "Learning rate array length does not match epochs"
assert len(R_FX) == EPOCHS, "R_FX loss ratio array length does not match epochs"
assert len(R_IY) == EPOCHS, "R_IY loss ratio array length does not match epochs"
assert len(R_BRZ) == EPOCHS, "R_BRZ loss ratio array length does not match epochs"
assert len(ACTIVE_LOSSES) == len(LOSS_NAMES), "ACTIVE_LOSSES length does not match LOSS_NAMES"

In [ ]:
# # test net I/O
# test_network_io()

In [ ]:
# plot schedulers
plt.figure(figsize=(12, 12))
plt.subplot(5, 1, 1) 
plt.plot(LR)
plt.ylim(0, np.max(LR)*1.1)
plt.title('Learning Rate')
plt.subplot(5, 1, 2)
plt.plot(R_FX)
plt.ylim(0, np.max(R_FX)*1.1)
plt.title('R_FX Loss Ratio')
plt.subplot(5, 1, 3)
plt.plot(R_IY)
plt.ylim(0, np.max(R_IY)*1.1)
plt.title('R_IY Loss Ratio')
plt.subplot(5, 1, 4)
plt.plot(R_BRZ)
plt.ylim(0, np.max(R_BRZ)*1.1)
plt.title('R_BRZ Loss Ratio')
plt.subplot(5, 1, 5)
plt.plot(R_PIN)
plt.ylim(0, np.max(R_PIN)*1.1)
plt.title('R_PIN Loss Ratio')
plt.tight_layout()
plt.show() if LOCAL else plt.savefig(f"{SAVE_DIR}/schedulers.png")

In [ ]:
# create training and evaluation datasets
train_ds = LiuqeDataset(TRAIN_DS_PATH)
val_ds = LiuqeDataset(EVAL_DS_PATH)
test_dataset(val_ds)
#save the mean_std of the dataset int the save directory
np.savez(f"{SAVE_DIR}/x_mean_std.npz", x_mean_std=train_ds.x_mean_std.cpu().numpy())

## Training

In [ ]:
def train():
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True) # initialize DataLoader
    val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False) # initialize DataLoader
    # network
    model = liuqeML(train_ds.x_mean_std)
    opt = torch.optim.Adam(model.parameters(), lr=LR[0])# adam
    # opt = torch.optim.AdamW(model.parameters(), lr=LR[0], weight_decay=1e-4) # adamw
    # opt = torch.optim.Muon(model.parameters(), lr=LR[0]) # Muon optimizer

    if LOAD_PRETRAINED is not None: # load pretrained model
        model.load_state_dict(torch.load(LOAD_PRETRAINED, map_location=torch.device(CPU)), strict=STRICT_LOAD) # load pretrained model
        print(f"Pretrained model loaded: {LOAD_PRETRAINED}")

    model.to(DEV) # move model to DEV
    loss_fn = F.mse_loss # Mean Squared Error
    # loss_fn = F.l1_loss # Mean Absolute Error
    tlog, elog = np.inf*np.ones((EPOCHS, len(train_dl), len(LOSS_NAMES))), np.inf*np.ones((EPOCHS, len(val_dl), len(LOSS_NAMES))) # init log
    start_time = time() # start time
    print("Ep:    || fx    | iy  | brz | brzp|| r:fx| r:iy|r:brz|r:pin| LR  || t  | eta| improved    ")
    for ep in range(EPOCHS): # epochs
        epoch_time = time()
        for pg in opt.param_groups: pg['lr'] = LR[ep] # update learning rate
        model.train()
        for ib, batch in enumerate(train_dl):
            if train_ds.on_dev: x, p, fx, iy, br, bz = batch # unpack batch
            else: x, p, fx, iy, br, bz = map(lambda t: t.to(DEV), batch) # move to DEV

            # PINN version
            r = p.clone()[:,:,0] # r coordinates
            p.requires_grad = True # enable gradients for p

            opt.zero_grad() # zero gradients
            fxp, brzp, iyp = model(x, p) # forward pass
            brp, bzp = brzp[:,:,0], brzp[:,:,1] # unpack into br and bz

            # losses
            l_fx = loss_fn(fx, fxp) # fx loss
            l_iy = loss_fn(iy, iyp) # iy loss
            l_brz = loss_fn(br, brp) + loss_fn(bz, bzp) # br bz loss

            # calculate derivatives of Fx wrt r and z 
            dFx_drz = torch.autograd.grad(
                outputs=fxp, inputs=p,
                grad_outputs=torch.ones_like(fxp),
                create_graph=True
            )
            dF_dr = dFx_drz[0][:,:,0] # derivative of Fx wrt r
            dF_dz = dFx_drz[0][:,:,1] # derivative of Fx wrt z
            Br_pinn = -dF_dz / (2 * torch.pi * r) # Br from derivative
            Bz_pinn = dF_dr / (2 * torch.pi * r) # Bz from derivative
            l_brz_pinn = loss_fn(br, Br_pinn) + loss_fn(bz, Bz_pinn) # br bz loss from derivative

            loss = R_FX[ep]*l_fx + R_IY[ep]*l_iy + R_BRZ[ep]*l_brz + R_PIN[ep]*l_brz_pinn # total loss
            loss.backward() # backprop

            # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0) # gradient clipping
            
            opt.step() # update weights

            # tlog[ep, ib] = (l_fx.item(), l_iy.item(), l_br.item(), l_bz.item(), l_br_pinn.item(), l_bz_pinn.item()) # save batch losses
            tlog[ep, ib] = (l_fx.item(), l_iy.item(), l_brz.item(), l_brz_pinn.item()) # save batch losses

        model.eval() # evaluation mode
        for ib, batch in enumerate(val_dl):
            if val_ds.on_dev: x, p, fx, iy, br, bz = batch # unpack batch
            else: x, p, fx, iy, br, bz = map(lambda t: t.to(DEV), batch) # move to DEV

            # PINN version
            r = p.clone()[:,:,0] # r coordinates
            p.requires_grad = True # enable gradients for p

            fxp, brzp, iyp = model(x, p) # forward pass
            brp, bzp = brzp[:,:,0], brzp[:,:,1] # unpack outputs

            dFx_drz = torch.autograd.grad(outputs=fxp.sum(), inputs=p) # compute gradients wrt p
            dF_dr = dFx_drz[0][:,:,0] # derivative of Fx wrt r
            dF_dz = dFx_drz[0][:,:,1] # derivative of Fx wrt z
            Br_pinn = -dF_dz / (2 * torch.pi * r) # Br from derivative
            Bz_pinn = dF_dr / (2 * torch.pi * r) # Bz from derivative

            # losses
            l_fx = loss_fn(fx, fxp) # fx loss
            l_iy = loss_fn(iy, iyp) # iy loss
            l_brz = loss_fn(br, brp) + loss_fn(bz, bzp) # br bz loss
            l_brz_pinn = loss_fn(br, Br_pinn) + loss_fn(bz, Bz_pinn) # br bz loss from derivative

            # elog[ep, ib] = (l_fx.item(), l_iy.item(), l_br.item(), l_bz.item(), l_br_pinn.item(), l_bz_pinn.item())
            elog[ep, ib] = (l_fx.item(), l_iy.item(), l_brz.item(), l_brz_pinn.item())
        # save model if improved
        endp = ""
        epoch_losses = np.mean(elog[ep,:,:], axis=0) # epoch losses: mean across batchess
        best_losses = np.min(np.mean(elog[:ep,:,:], axis=1), axis=0) if ep > 0 else epoch_losses # best losses: min across epochs
        for el, bl, n in zip(epoch_losses, best_losses, LOSS_NAMES):
            if el <= bl: torch.save(model.state_dict(), model_path(n)); endp+=f"*{n}"
        # print progress
        print(f"{ep+1:03d}/{EPOCHS:02d}||" + f"{epoch_losses[0]:.1e}|" +
            '|'.join([f"{v:.0e}" for v in epoch_losses[1:]]) + "||" +
            f"{R_FX[ep]:.0e}|{R_IY[ep]:.0e}|{R_BRZ[ep]:.0e}|{R_PIN[ep]:.0e}|{LR[ep]:.0e}||" +
            f"{int(time()-epoch_time):03d}s|{int((time()-start_time)*(EPOCHS-ep)/(ep+1)/60):03d}m|", end=endp+'\n')
        # if ep >= 10 and best_losses[2] > 0.1: return False 

    tlosses, elosses = np.mean(tlog, axis=1), np.mean(elog, axis=1) # losses across epochs
    print(f"Training time: {(time()-start_time)/60:.0f}mins")
    print("Best MSE: " + ", ".join([f"{n} {bl:.3e} (ep {be})" for n, bl, be in zip(LOSS_NAMES, np.min(elosses, axis=0), np.argmin(elosses, axis=0))]))
    print("Estimated MAE: " + ", ".join([f"{n} {np.sqrt(np.min(np.mean(elog[:,:,i], axis=1))):.3e}" for i, n in enumerate(LOSS_NAMES)]))
    np.save(f"{SAVE_DIR}/train_losses.npy", tlosses) # save training losses
    np.save(f"{SAVE_DIR}/eval_losses.npy", elosses) # save evaluation losses
    return True

# train the model (multiple attempts)
for i in range(20): 
    print('Starting training...')
    success = train()
    if success: break
    else: print(f"Convergence failed, retrying... {i+1}/20")
# if not success: delete the files and exit:
if not success: os.system(f"rm -rf {SAVE_DIR}")
assert success, "Training failed, no model saved"

In [ ]:
# plot losses
print("Plotting losses...")
fig, ax = plt.subplots(2, len(LOSS_NAMES), figsize=(4*len(LOSS_NAMES), 8))
# Load losses
train_loss = np.load(f"{SAVE_DIR}/train_losses.npy")
eval_loss = np.load(f"{SAVE_DIR}/eval_losses.npy")
for i, name in enumerate(LOSS_NAMES):
    # Linear scale
    ax[0,i].plot(train_loss[:,i], label='train')# y1_mean_std = np.concatenate([μy1, Σy1], axis=0)
# y2_mean_std = np.concatenate([μy2, Σy2], axis=0)
    ax[0,i].plot(eval_loss[:,i], label='eval')
    ax[0,i].set_title(f"{name}")
    ax[0,i].set_xlabel("Epoch")
    ax[0,i].set_ylabel("Loss")
    ax[0,i].legend(); ax[0,i].grid(True)
    # Log scale
    ax[1,i].plot(train_loss[:,i], label='train')
    ax[1,i].plot(eval_loss[:,i], label='eval')
    ax[1,i].set_yscale('log')
    ax[1,i].set_title(f"{name} (log)")
    ax[1,i].set_xlabel("Epoch")
    ax[1,i].set_ylabel("Loss")
    ax[1,i].legend(); ax[1,i].grid(True)
plt.tight_layout()
plt.show() if LOCAL else plt.savefig(f"{SAVE_DIR}/losses.png")

In [ ]:
# testing network outputs
# try:
if True:
    print("Testing network output...")
    for active, ln in zip(ACTIVE_LOSSES, LOSS_NAMES):
        if not active: continue
        model = liuqeML(train_ds.x_mean_std)
        model.load_state_dict(torch.load(model_path(ln), map_location=torch.device(CPU))) # load pretrained model
        plot_network_outputs(val_ds, model, title=ln, pinn=R_PIN[-1]>R_BRZ[-1]) # plot network outputs
# except Exception as e:
#     print(f"Error in testing network output: {e}")
#     print("Skipping network output testing...")
#     pass

In [ ]:
# save weights to .mat file
m = liuqeML(train_ds.x_mean_std)
for active, ln in zip(ACTIVE_LOSSES, LOSS_NAMES):
    if active:
        m.load_state_dict(torch.load(model_path(ln), map_location=torch.device(CPU))) # load pretrained model
        save_weights_to_mat(m, title=ln)

In [ ]:
print(f"{JOBID} done")
if not LOCAL: sleep(30) # wait for files to update (for cluster)
#copy the log file to the folder
os.system(f"cp jobs/{JOBID}.txt {SAVE_DIR}/log.txt")